# Assignment 5 — Can We Run This On Our Own Hardware?

**ISM 6564 — Text Analytics · Fall 2026 · Week 5: Transformers & Language Models**

Northwind Outfitters' support desk receives about **2,400 tickets a week**. Every one is
currently read by a human and dropped into one of six queues. The support operations
director wants the first pass automated, and has two proposals on the table:

1. **Call a frontier model's API** for every ticket.
2. **Run a small open-weights model on the team's own hardware** — the Mac Studio already
   sitting in the ops room — so ticket text never leaves the building.

Option 2 is on the table for a specific reason: tickets contain order numbers, partial card
digits and home addresses, and Legal has questions about sending that to a third party. If a
local model is good enough, that question disappears. "Good enough" is what you are being
asked to measure.

> **Goal:** put a small language model on your own machine and find out, in numbers, what it
> costs to run, what it does with a customer's message, and how well it routes the desk's
> tickets — then write the recommendation. Every task is one lecture slide turned into a
> measurement you make yourself.

**Everything you need is in this notebook:** the deadlines, what to submit, how it is marked,
the setup notes and every task. The data loads itself from the course's public data
repository, so there is nothing to download by hand.


## When and how

- **Opens:** Wednesday, September 23, 2026 at 12:30 PM ET (start of Week 5 class).
- **Due:** Tuesday, September 29, 2026 at 11:59 PM ET (the evening before Week 6's class).
- **Late work:** accepted up to 24 hours late with a **20% penalty**. After Wednesday, September 30, 2026 at 11:59 PM ET it is
  not accepted and scores zero.
- **This assignment counts toward your assignment grade** (30% of the course total, spread
  over ten assignments).

### What to submit

Submit **two** items to Canvas before the deadline:

1. **This notebook**, with every cell run from the top and the outputs visible.
2. **Your AI Usage Report**, written on the course template:
   [templates/ai-usage-report.md](https://github.com/prof-tcsmith/6564F26-DATA/blob/main/templates/ai-usage-report.md).
   AI use is required on every assignment, so the report is never optional. It needs the
   tool and model version, the **complete transcript** of your AI conversations (not a summary
   or a share link), and a short note on how you checked the output. See §7 of the syllabus
   (on Canvas).


## How it is marked

| Component | Points |
| --- | ---: |
| **Task 1** — helpers pass the self-check (6), memory and cache tables produced for all three models (6), written answer correct on the arithmetic, the 8 GB laptop, the long-prompt speed-up and which bill grows (8) | 20 |
| **Task 2** — `reply` correct, returning only the new text (5); both provided cells run with readable output (3); written answer correct on the two stages, greedy vs sampled, temperature 1.5, and the route / draft choice (7) | 15 |
| **Task 3** — `parse_label` passes the self-check and `classify_all` runs all three models (10); metrics, per-queue table and confusion matrix present (6); error analysis grounded in actual raw replies and the tie-break rules, speed explained from tokens, costliest mistake named (14) | 30 |
| **Task 4** — `usd_cost` correct and the comparison table complete (8); memo: a committed choice with ≥4 measured numbers (8), evaluation with fixture-vs-measurement stated and a next measurement named (7), control with flip conditions (7) | 30 |
| **Reproducibility and presentation** — seeds untouched, notebook runs top to bottom, outputs visible, no stray scratch cells | 5 |
| **AI Usage Report** (complete: tool statement, full logs, validation note) | 0 (gate: −5 to −10 if incomplete, up to −20 if missing) |
| **Total** | **100** |


## How to work through this notebook

- Every place you have to write something is marked `# TODO`. Everything else — imports,
  loading, the chat-prompt helper, the metric tables, the cost constants — is written for
  you.
- **The notebook runs top to bottom right now, with nothing filled in.** Cells that depend
  on an unfinished `# TODO` print `SKIPPED` and move on. As you fill them in, more of the
  notebook comes alive. Re-run from the top when you finish a task.
- Markdown cells headed **Your answer** are graded. Replace the placeholder text. Tasks 3
  and 4 put more marks on the prose than on the code.
- Do not change `SEED`, and do not change `GREEDY`.

Every task below names the lecture slide it comes from. If a step feels unfamiliar, open the
companion notes for that slide (on Canvas) and the lecture notebook section beside it before
you ask an AI assistant.

## Before you start

**Do the Week 5 practice notebook first** (on Canvas). It rehearses every mechanic this
assignment uses — the chat template, the generation loop, the cache timing, sampling with a
seed — on a 135M model that runs in seconds.

**The models.** Three instruction-tuned decoders, small enough for a laptop, plus the base
version of the middle one:

| Checkpoint | Learned numbers | Download | Used in |
| --- | ---: | ---: | --- |
| `HuggingFaceTB/SmolLM2-360M-Instruct` | ~362M | ~0.7 GB | Tasks 1, 3, 4 |
| `Qwen/Qwen2.5-0.5B-Instruct` | ~494M | ~1.0 GB | Tasks 1–4 |
| `Qwen/Qwen2.5-1.5B-Instruct` | ~1.54B | ~3.1 GB | Tasks 1, 3, 4 |
| `Qwen/Qwen2.5-0.5B` (base) | ~494M | ~1.0 GB | Task 2 |

- **The first run downloads about 6 GB** into `~/.cache/huggingface/hub`; after that
  everything is read from disk. Budget about 6 GB of free disk and do the download well before
  the night it is due. If disk space is a problem, drop the 1.5B model, say so in your
  notebook, and lose only the top rung of the size comparison.
- **Google Colab is fine.** Pick a T4 runtime; the models download in about a minute per
  session. Wherever this notebook says "your machine", name the Colab runtime instead.
- **Expect roughly 10–15 minutes of compute** for a filled-in notebook, most of it in Task 3.
  If one model's 60-ticket pass takes more than a few minutes, check that you are on the
  accelerator and not on the CPU.
- **Sanity anchors.** Greedy decoding is deterministic, so your accuracy numbers should
  reproduce exactly across runs on the same machine. Expect the largest model somewhere around
  0.6–0.7 and the smallest well below 0.3. If your largest model scores near chance (0.167),
  the most likely cause is that `build_messages` is not sending the system prompt.
- **You must be able to defend what you submit.** See §6 of the syllabus (on Canvas). If your
  AI assistant wrote `time_to_write`, be ready to explain what `use_cache=False` makes the
  model redo for every new token, and why the answer comes out the same either way.


## Setup

Seeds are set here for `random`, `numpy` and `torch`. Greedy decoding is deterministic
(lecture: *Choosing the next token*), so your **accuracy** numbers should reproduce exactly
on the same machine. Your **timings** will not reproduce, on your machine or anyone else's —
that is expected, and it is why the write-ups ask you to explain directions and ratios
rather than defend three decimal places.


In [1]:
# --- imports, seeds, device --------------------------------------------------
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"torch {torch.__version__}")
print(f"device: {device}")


torch 2.11.0+cpu
device: cpu


### The models, and the decoding settings

Three instruction-tuned decoders of three sizes — the lecture's size ladder, one rung
smaller at the bottom — plus the **base** version of the middle one, so Task 2 can show what
instruction tuning changed.

`GREEDY` pins the decoding settings. A checkpoint ships its own defaults (Qwen's turn
sampling on, with a temperature of 0.7), and `generate()` applies them unless you say
otherwise. Greedy means: always the most likely token, so the same prompt gives the same
text every time. Use `GREEDY` everywhere except where Task 2 tells you to sample.


In [2]:
# --- models and pinned decoding --------------------------------------------------
MODELS = [
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]
BASE = "Qwen/Qwen2.5-0.5B"          # the same weights as MODELS[1], before instruction tuning
DTYPE = torch.float32               # full precision: 4 bytes per learned number, as in the lecture

GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0)


def SAMPLE(temperature: float, top_p: float = 1.0) -> dict:
    """Sampling settings: temperature, optional top-p, and nothing else (top-k off)."""
    return dict(do_sample=True, temperature=temperature, top_p=top_p, top_k=0, repetition_penalty=1.0)


# What each checkpoint would do if you did NOT override it:
for _m in MODELS:
    _gc = GenerationConfig.from_pretrained(_m)
    print(f"{_m:42s} do_sample={_gc.do_sample}  temp={_gc.temperature}  "
          f"top_p={_gc.top_p}  top_k={_gc.top_k}  rep_pen={_gc.repetition_penalty}")


HuggingFaceTB/SmolLM2-360M-Instruct        do_sample=None  temp=None  top_p=None  top_k=None  rep_pen=None
Qwen/Qwen2.5-0.5B-Instruct                 do_sample=True  temp=0.7  top_p=0.8  top_k=20  rep_pen=1.1
Qwen/Qwen2.5-1.5B-Instruct                 do_sample=True  temp=0.7  top_p=0.8  top_k=20  rep_pen=1.1


In [3]:
# --- scaffolding so this notebook runs with TODOs unfilled -------------------
# You do not need to modify anything in this cell.

class Unfinished(Exception):
    """Raised by scaffolding you have not filled in yet."""


def todo(what: str):
    """Call from inside an unfinished TODO so the notebook still runs."""
    raise Unfinished(f"{what} is not implemented yet.")


def attempt(label, fn, *args, **kwargs):
    """Run `fn`; return None (with a message) if it hits an unfinished TODO."""
    try:
        return fn(*args, **kwargs)
    except Unfinished as exc:
        print(f"SKIPPED [{label}]  {exc}  Fill in the TODO above and re-run.")
        return None


def need(*values) -> bool:
    """True (with a message) if any earlier TODO is still unfilled."""
    if any(v is None for v in values):
        print("SKIPPED  an earlier `# TODO` is still unfilled. Finish it, then re-run this cell.")
        return True
    return False


def sync():
    """Block until the accelerator has actually finished. Timing is a lie without it."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def free_model(model=None):
    """Drop a model and hand the memory back, so four checkpoints fit in one run."""
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def load_model(name: str, dtype=DTYPE):
    """Provided. Returns (tokenizer, model) on `device`, in eval mode."""
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(device).eval()
    return tok, model


def chat_prompt(tok, user_text: str, system: str | None = None) -> str:
    """Provided. The exact string an instruct model expects: its chat template, applied.

    The lecture's *A decoder becomes an assistant* slide: an instruct model was tuned on
    conversations written in one exact format, so every message goes through the tokenizer's
    `apply_chat_template` rather than being sent as a bare string.
    """
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": user_text}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


print("scaffolding ready")


scaffolding ready


### Load the ticket set

`support_tickets.csv` holds 60 real-shaped support tickets, hand-labelled into six queues,
ten per queue. The cell below reads it from the course's public data repository, or from a
local `data/` folder if you have one, so it works on your laptop and in a Colab tab with
nothing downloaded by hand.

| Column | Meaning |
| --- | --- |
| `id` | `T001` … `T060`. |
| `category` | Gold label: `Shipping`, `Billing`, `Returns`, `Defect`, `Account`, `Other`. |
| `difficulty` | `clear` (7 per queue) or `ambiguous` (3 per queue). |
| `text` | The customer's message. |

An `ambiguous` ticket is one where a careful human annotator has to apply a tie-break rule
rather than read the answer off the surface: *"the stove failed on day two of a trip and I
want my money back"* is a fault report (`Defect`) under the desk's rules, but it is also
plainly a refund request. The desk's tie-break rules are in Task 3.


In [4]:
# --- load ---------------------------------------------------------------------
DATA_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-05-transformers-llms/data/")
CANDIDATES = [Path("data"), Path("week-05-transformers-llms/data"),
              Path("../data"), Path("../../week-05-transformers-llms/data")]


def data_path(name: str) -> str:
    for base in CANDIDATES:
        if (base / name).exists():
            return str(base / name)
    return DATA_URL + name


tickets = pd.read_csv(data_path("support_tickets.csv"))
GOLD = tickets.category.tolist()
CATEGORIES = ["Shipping", "Billing", "Returns", "Defect", "Account", "Other"]

assert len(tickets) == 60
assert set(tickets.category) == set(CATEGORIES)

print(f"{len(tickets)} tickets")
print(pd.crosstab(tickets.category, tickets.difficulty).loc[CATEGORIES])
print()
print(tickets.head(3).to_string(index=False, max_colwidth=70))


60 tickets
difficulty  ambiguous  clear
category                    
Shipping            3      7
Billing             3      7
Returns             3      7
Defect              3      7
Account             3      7
Other               3      7

  id category difficulty                                                                   text
T001 Shipping      clear Order 88213 was supposed to arrive Tuesday and the tracking hasn't ...
T002 Shipping      clear Tracking says delivered but there is nothing at my door and nothing...
T003 Shipping      clear I paid for two-day shipping on Thursday and the order still hasn't ...


In [5]:
# --- the routing prompt every model sees ---------------------------------------
# Identical for all four systems under comparison. If the prompt differs, the
# comparison measures the prompt, not the model.

SYSTEM_PROMPT = (
    "You are a support-ticket triage assistant for Northwind Outfitters, an online "
    "outdoor-gear retailer.\n"
    "Read the customer message and choose the single best category.\n\n"
    "Categories:\n"
    "- Shipping: where is my order, delivery delays, lost or misdelivered packages, "
    "tracking, delivery address.\n"
    "- Billing: charges, refunds not yet received, invoices, payment methods, promo "
    "codes, tax.\n"
    "- Returns: starting a return or exchange, return labels, return policy, wrong "
    "item or size sent.\n"
    "- Defect: the product arrived damaged or failed in use; quality complaints.\n"
    "- Account: login, password, email address, profile, loyalty points, newsletter "
    "subscription.\n"
    "- Other: anything that does not fit the categories above.\n\n"
    "Answer with exactly one category word and nothing else."
)
print(SYSTEM_PROMPT)


You are a support-ticket triage assistant for Northwind Outfitters, an online outdoor-gear retailer.
Read the customer message and choose the single best category.

Categories:
- Shipping: where is my order, delivery delays, lost or misdelivered packages, tracking, delivery address.
- Billing: charges, refunds not yet received, invoices, payment methods, promo codes, tax.
- Returns: starting a return or exchange, return labels, return policy, wrong item or size sent.
- Defect: the product arrived damaged or failed in use; quality complaints.
- Account: login, password, email address, profile, loyalty points, newsletter subscription.
- Other: anything that does not fit the categories above.

Answer with exactly one category word and nothing else.


### Smoke test — does this machine actually do this?

Provided, and not behind a TODO on purpose. It downloads the smallest of the checkpoints
and generates thirty tokens. If this cell is slow the first time, that is the download; if
it is slow every time, check `device` above.

Read the reply. It is a 362-million-number model, and the answer it gives is your first
evidence about what that buys you.


In [6]:
# --- smoke test ---------------------------------------------------------------
_tok, _model = load_model(MODELS[0])
_prompt = chat_prompt(_tok, "In one sentence: what does an online retailer's support desk do?",
                      system="You are a concise assistant.")
_ids = _tok(_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    _out = _model.generate(**_ids, max_new_tokens=30, pad_token_id=_tok.eos_token_id, **GREEDY)
print(_tok.decode(_out[0][_ids["input_ids"].shape[1]:], skip_special_tokens=True))
free_model(_model)
del _tok, _model, _ids, _out


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

An online retailer's support desk provides assistance to customers with issues related to their online shopping experience.


---

## Task 1 — Two bills, before you download anything

*Lecture slide: **Running a model: two bills, memory and time**. Notebook §4.4.*

**Bill 1, memory to hold the model,** is one multiplication: learned numbers × bytes per
number. **Bill 2, time to write,** is one token at a time, each new token looking back at the
whole text so far — unless the library *remembers* the past instead of re-reading it, which
it may, because a decoder's past never changes.

Three helpers to write, then two provided cells that run them over the three models.


In [7]:
# --- 1a: the memory bill -----------------------------------------------------

def learned_numbers(model) -> int:
    """How many learned numbers the model holds.

    Every parameter tensor `p` in `model.parameters()` holds `p.numel()` numbers; add them up.
    """
    return sum(p.numel() for p in model.parameters())


def predicted_gb(n_numbers: int, bytes_per_number: float) -> float:
    """Bill 1, predicted: numbers x bytes per number, in gigabytes (divide by 1e9)."""
    return n_numbers * bytes_per_number / 1e9


def measured_gb(model) -> float:
    """Bill 1, measured: the bytes the parameters actually occupy right now, in gigabytes.

    Each tensor occupies `p.numel() * p.element_size()` bytes (element_size is 4 at full
    precision, 2 at half). Add them up and divide by 1e9.
    """
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9


# quick self-check on a tiny stand-in "model": two tensors, 10 and 6 numbers, 4 bytes each
_probe = torch.nn.Sequential(torch.nn.Linear(2, 3), torch.nn.Linear(3, 2))     # 9 + 8 = 17 numbers
if attempt("learned_numbers", learned_numbers, _probe) is not None:
    assert learned_numbers(_probe) == 17, learned_numbers(_probe)
    assert abs(predicted_gb(8_000_000_000, 4) - 32.0) < 1e-9
    assert abs(measured_gb(_probe) - 17 * 4 / 1e9) < 1e-12
    print("Task 1a helpers pass their self-check")


Task 1a helpers pass their self-check


In [8]:
# --- provided: Bill 1 for the three models ---------------------------------------------
# Each model is loaded at full precision (4 bytes), measured, and freed. Then the middle
# model is loaded again at HALF precision (2 bytes), so you can see the second row of the
# lecture's table come true on your own machine.
memory_rows = []
for _name in MODELS:
    _tok, _model = load_model(_name)
    _n = attempt("learned_numbers", learned_numbers, _model)
    if _n is None:
        free_model(_model); del _tok, _model
        break
    memory_rows.append({
        "model": _name.split("/")[-1],
        "learned numbers (M)": round(_n / 1e6, 1),
        "predicted GB @ 4 bytes": round(predicted_gb(_n, 4), 2),
        "predicted GB @ 2 bytes": round(predicted_gb(_n, 2), 2),
        "predicted GB @ 1/2 byte (4-bit)": round(predicted_gb(_n, 0.5), 2),
        "measured GB (as loaded)": round(measured_gb(_model), 2),
    })
    free_model(_model); del _tok, _model

memory_table = pd.DataFrame(memory_rows) if len(memory_rows) == len(MODELS) else None
if memory_table is not None:
    print(memory_table.to_string(index=False))
    _tok, _half = load_model(MODELS[1], dtype=torch.bfloat16)
    print(f"\n{MODELS[1].split('/')[-1]} loaded at HALF precision: measured {measured_gb(_half):.2f} GB"
          f"  (predicted @ 2 bytes: {memory_table.loc[1, 'predicted GB @ 2 bytes']:.2f} GB)")
    free_model(_half); del _tok, _half


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

                model  learned numbers (M)  predicted GB @ 4 bytes  predicted GB @ 2 bytes  predicted GB @ 1/2 byte (4-bit)  measured GB (as loaded)
SmolLM2-360M-Instruct                361.8                    1.45                    0.72                             0.18                     1.45
Qwen2.5-0.5B-Instruct                494.0                    1.98                    0.99                             0.25                     1.98
Qwen2.5-1.5B-Instruct               1543.7                    6.17                    3.09                             0.77                     6.17


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Qwen2.5-0.5B-Instruct loaded at HALF precision: measured 0.99 GB  (predicted @ 2 bytes: 0.99 GB)


In [9]:
# --- 1b: the time bill --------------------------------------------------------

def time_to_write(model, tok, prompt: str, n_new: int = 32, use_cache: bool = True) -> dict:
    """Write exactly `n_new` tokens after `prompt`, greedily, and time it.

    Returns {"prompt_tokens": int, "seconds": float, "tokens_per_second": float}.

    Steps
      1. ids = tok(prompt, return_tensors="pt").to(device)
      2. warm up: one short generate (4 tokens) so the first call's start-up cost is not timed
      3. sync(); t0 = time.perf_counter()
      4. model.generate(**ids, max_new_tokens=n_new, min_new_tokens=n_new, use_cache=use_cache,
                        pad_token_id=tok.eos_token_id, **GREEDY)   -- inside torch.no_grad()
      5. sync(); seconds = time.perf_counter() - t0
      6. return the dict; prompt_tokens is ids["input_ids"].shape[1]

    `use_cache=True` is the library remembering the past; `use_cache=False` makes it
    re-read the whole text for every new token.
    """
    ids = tok(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        model.generate(**ids, max_new_tokens=4, use_cache=use_cache,
                        pad_token_id=tok.eos_token_id, **GREEDY)

    sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(**ids, max_new_tokens=n_new, min_new_tokens=n_new, use_cache=use_cache,
                        pad_token_id=tok.eos_token_id, **GREEDY)
    sync()
    seconds = time.perf_counter() - t0

    return {
        "prompt_tokens": ids["input_ids"].shape[1],
        "seconds": seconds,
        "tokens_per_second": n_new / seconds,
    }


In [10]:
# --- provided: Bill 2 -- one token at a time, remembering the past or not ---------------
# A short prompt (one ticket) and a long one (the same ticket after a long transcript),
# each written to 32 new tokens, with the past remembered and with it re-read every step.
FILLER = ("Earlier in this conversation the customer described a delayed order, the agent "
          "quoted the carrier's tracking status and the exception code, and the customer "
          "asked twice when the parcel would be rescanned at the regional hub. ")
TICKET = tickets.loc[tickets.id == "T002", "text"].item()

if not need(attempt("time_to_write", lambda: time_to_write)):
    _tok, _model = load_model(MODELS[1])
    SHORT = chat_prompt(_tok, TICKET + "\n\nReply in two sentences.", system="You are a support agent.")
    LONG = chat_prompt(_tok, FILLER * 22 + "\n\n" + TICKET + "\n\nReply in two sentences.",
                       system="You are a support agent.")
    cache_rows = []
    for _label, _prompt in (("short", SHORT), ("long", LONG)):
        on = attempt("time_to_write", time_to_write, _model, _tok, _prompt, 32, True)
        off = attempt("time_to_write", time_to_write, _model, _tok, _prompt, 32, False)
        if on is None or off is None:
            cache_rows = None
            break
        cache_rows.append({"text so far": f"{_label} ({on['prompt_tokens']:,} tokens)",
                           "re-reads everything (s)": round(off["seconds"], 2),
                           "remembers the past (s)": round(on["seconds"], 2),
                           "speed-up": f"{off['seconds'] / on['seconds']:.1f}x",
                           "tokens/s with the past remembered": round(on["tokens_per_second"], 1)})
    free_model(_model); del _tok, _model
    if cache_rows:
        cache_table = pd.DataFrame(cache_rows)
        print(f"{MODELS[1].split('/')[-1]}, 32 new tokens\n")
        print(cache_table.to_string(index=False))

        # and the writing speed of all three sizes, on the short prompt, past remembered
        speed_rows = []
        for _name in MODELS:
            _tok, _model = load_model(_name)
            r = time_to_write(_model, _tok, chat_prompt(_tok, TICKET + "\n\nReply in two sentences.",
                                                        system="You are a support agent."), 32, True)
            speed_rows.append({"model": _name.split("/")[-1], "tokens/s": round(r["tokens_per_second"], 1)})
            free_model(_model); del _tok, _model
        speed_table = pd.DataFrame(speed_rows)
        print()
        print(speed_table.to_string(index=False))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2.5-0.5B-Instruct, 32 new tokens

      text so far  re-reads everything (s)  remembers the past (s) speed-up  tokens/s with the past remembered
short (40 tokens)                    10.57                    2.31     4.6x                               13.8
long (943 tokens)                    91.22                    5.63    16.2x                                5.7


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


                model  tokens/s
SmolLM2-360M-Instruct      19.6
Qwen2.5-0.5B-Instruct      17.4
Qwen2.5-1.5B-Instruct       7.3


### Your answer — Task 1


0.5B means about half a billion learned numbers, not half a GB. At full precision each number takes 4 bytes, so Qwen2.5-0.5B-Instruct (494.0M numbers) takes 1.98 GB. At half precision it takes 2 bytes per number, which takes 0.99 GB, which matches the prediction. At full precision all three models fit on an 8 GB laptop (1.45, 1.98 and 6.17 GB), but the 1.5B model leaves under 2 GB for the operating system and the cache, so it is a tight fit and may be slow. Storing each number in half a byte (4 bit) would cut them to 0.18, 0.25 and 0.77 GB, which is much smaller.

The speed up from remembering the past is 4.6x on the 40 token prompt and 16.2x on the 943 token prompt. Without the cache, every new token makes the model reread the whole text so far (prompt plus everything already written), so the work per token grows with length, 91.22s vs 10.57s. With the cache, the model keeps what it computed from earlier tokens and processes only the one new token, so the cost grows slowly (2.31s to 5.63s). A decoder is allowed to do this because each position only looks at earlier positions. The time bill grows with the length of the conversation, while the memory bill for the weights is fixed once you pick a model and precision.

---

## Task 2 — What the model does with a message

*Lecture slides: **A decoder becomes an assistant in two stages** and **Choosing the next
token**. Notebook §4.2 and §5.7.*

One helper to write — the four-line loop from the lecture, wrapped by `generate` — then two
provided cells. The first hands the same ticket to the **base** model and to the **instruct**
model built from it. The second hands one prompt to the instruct model under five decoding
settings.


In [11]:
# --- 2a: get a reply -----------------------------------------------------------

def reply(model, tok, prompt: str, max_new_tokens: int = 60, **decoding) -> str:
    """Tokenize `prompt`, generate up to `max_new_tokens`, return ONLY the new text.

    `decoding` is either GREEDY or one of the SAMPLE(...) dicts.

    Steps
      1. ids = tok(prompt, return_tensors="pt").to(device)
      2. out = model.generate(**ids, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.eos_token_id, **decoding)   -- inside torch.no_grad()
      3. new = out[0][ids["input_ids"].shape[1]:]     -- the tokens AFTER the prompt
      4. return tok.decode(new, skip_special_tokens=True).strip()
    """
    ids = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.eos_token_id, **decoding)
    new = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip()


In [12]:
# --- provided: base vs instruct, same weights before and after the second stage ---------
# The base model is a text continuer: it gets the ticket as bare text. The instruct model
# gets the same ticket through its chat template.
if not need(attempt("reply", lambda: reply)):
    _btok, _base = load_model(BASE)
    print(f"TICKET: {TICKET}\n")
    print(f"--- {BASE}  (bare text in, greedy) ---")
    _text = attempt("reply", reply, _base, _btok, TICKET + "\n\n", 60, **GREEDY)
    if _text is not None:
        print(_text)
    free_model(_base); del _btok, _base

    _tok, _inst = load_model(MODELS[1])
    print(f"\n--- {MODELS[1]}  (chat template, greedy) ---")
    _text = attempt("reply", reply, _inst, _tok, chat_prompt(_tok, TICKET, system="You are a support agent."),
                    60, **GREEDY)
    if _text is not None:
        print(_text)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

TICKET: Tracking says delivered but there is nothing at my door and nothing with my neighbours.

--- Qwen/Qwen2.5-0.5B  (bare text in, greedy) ---
Does it follow that "I have a lot of mail to deliver."?
Options are:
 (a). yes
 (b). it is not possible to tell
 (c). no
(b).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


--- Qwen/Qwen2.5-0.5B-Instruct  (chat template, greedy) ---
I'm sorry to hear that you're experiencing this issue. It sounds like there may be a problem with the delivery service or the delivery person's location. Here are a few steps you can take to try and resolve the issue:

1. Check the delivery service's website or app for any updates or


In [13]:
# --- provided: five ways of choosing the next token --------------------------------------
# Same model, same prompt. Greedy twice; temperature 0.7 with top-p 0.9 under two seeds;
# temperature 1.5 with no truncation.
if not need(attempt("reply", lambda: reply)):
    DRAFT = chat_prompt(_tok, "Write a two-sentence reply to this customer:\n\n" + TICKET,
                        system="You are a Northwind Outfitters support agent.")
    runs = [("greedy, run 1", GREEDY, None), ("greedy, run 2", GREEDY, None),
            ("T=0.7, top-p 0.9, seed 1", SAMPLE(0.7, 0.9), 1),
            ("T=0.7, top-p 0.9, seed 2", SAMPLE(0.7, 0.9), 2),
            ("T=1.5, no truncation, seed 1", SAMPLE(1.5, 1.0), 1)]
    for label, settings, seed in runs:
        if seed is not None:
            torch.manual_seed(seed)
        text = attempt("reply", reply, _inst, _tok, DRAFT, 60, **settings)
        if text is None:
            break
        print(f"[{label}]\n{text}\n")
    free_model(_inst); del _tok, _inst


[greedy, run 1]
I'm sorry to hear that you're experiencing issues with your order. Please let me know if you need any assistance with tracking or if there's anything else I can help you with.

[greedy, run 2]
I'm sorry to hear that you're experiencing issues with your order. Please let me know if you need any assistance with tracking or if there's anything else I can help you with.

[T=0.7, top-p 0.9, seed 1]
I'm sorry to hear that. Can you provide me with your contact information so I can assist you further?

[T=0.7, top-p 0.9, seed 2]
I apologize, but I don't have access to your specific delivery information. Can you please provide me with the tracking number so I can check for delivery status?

[T=1.5, no truncation, seed 1]
I'm sorry to hear that the product is failing to empty without their grasp itDMETHOD corporation design product efficiently prioritize detail一点点 caputo customer detail limeos bc related"wcu proof with discretion point合法checkBoxTexasample opti Production Burger社团

### Your answer — Task 2

The base model treated the ticket as text to continue. It became like a quiz ("Does it follow that, Options are: a) yes...") and never tried to help. The instruct model, with the same weights count and architecture, answered as an assistant, apologising and listing steps to fix the delivery problem. The second stage of training, which was tuning on conversations written in the chat template, changed what the model does with a prompt.

Greedy decoding always takes the single most likely next token, so with the same prompt it only produces the same text, which is why runs 1 and 2 match. Sampling draws each token at random from the distribution, so different draws give different replies. The seed fixes the random number generator, so the same seed replays the same draws and different seeds give different text. At T=1.5 with no truncation the distribution is flattened and the unlikely tail is fully allowed, so odd tokens get picked. Each odd token then becomes context for the next, and errors compound, and the reply decays into a jumble of unrelated words and languages.

For routing I would use greedy, because the same ticket must always get the same queue, and a label is one short answer that needs no variety. For drafting replies I would use T=0.7 with top p 0.9, because the wording sounds natural and varied while the truncation keeps it coherent.


---

## Task 3 — Does it actually do the job?

*Lecture slide: **Small vs large is an engineering decision**. Notebook §6.2–6.3.*

The lecture measured three model sizes on a 24-task set. Here you do the same on the desk's
own tickets: 60 of them, hand-labelled into six queues, ten per queue, with a `difficulty`
column that separates the ones a careful reader answers on sight (`clear`) from the ones that
need the desk's tie-break rules (`ambiguous`). **Read the rules before you argue with a label**
— "the model disagreed with the gold label" and "the gold label is wrong" are different
findings, and Task 3 asks you to tell them apart.

| Rule | Why |
| --- | --- |
| **Defect beats Returns.** A message reporting a product fault is `Defect` even when the customer asks for a refund or a return. | The Defect queue owns the warranty decision; routing it to Returns loses the fault report. |
| **Shipping beats Billing.** If the main ask is where an order is, it is `Shipping` even if a charge is mentioned. | Finding the parcel usually resolves the charge complaint too. |
| **Returns before the goods move, Billing after.** "I want to send this back" is `Returns`; "I returned it and my money has not arrived" is `Billing`. | The hand-off point is the carrier scan. |
| **Account covers credentials, profile and loyalty only.** A billing address stored in the profile is `Account`; a charge to a card is `Billing`. | Splits identity from money. |
| **Other is pre-sale, corporate and out-of-scope.** Product advice before purchase, sponsorship, careers, store hours, wholesale, repairs. | Not an order issue, so no order queue owns it. |

Two helpers to write, then the loop from the lecture notebook's `run_task_set`, then
provided tables.


In [14]:
# --- 3a: the message, and the parser -----------------------------------------------

def build_messages(text: str) -> list[dict]:
    """The two-message chat sent to every model: SYSTEM_PROMPT, then the ticket text."""
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]


def parse_label(raw: str):
    """Turn a reply into one of CATEGORIES, or None if no category word appears.

    Case-insensitive; the FIRST category word that appears anywhere in the reply wins
    ("Category: Billing." -> "Billing"). A reply with no category word is invalid: return
    None. In production an invalid reply cannot be routed, so it counts as wrong.
    """
    best_cat, best_pos = None, None
    for cat in CATEGORIES:
        match = re.search(r"\b" + re.escape(cat) + r"\b", raw, re.IGNORECASE)
        if match and (best_pos is None or match.start() < best_pos):
            best_cat, best_pos = cat, match.start()
    return best_cat


for _s, _want in [("Shipping", "Shipping"), (" billing ", "Billing"), ("**Account**", "Account"),
                  ("Category: Returns.", "Returns"), ("This is a Defect, not Billing", "Defect"),
                  ("I'm sorry to hear that", None), ("", None)]:
    _got = attempt("parse_label", parse_label, _s)
    if _got is None and _want is not None:
        break
    assert _got == _want, (_s, _got, _want)
else:
    print("parse_label passes its self-check")


parse_label passes its self-check


In [15]:
# --- 3b: run one model over all sixty tickets ------------------------------------------

def classify_all(model, tok, max_new_tokens: int = 8) -> dict:
    """Greedy-classify every ticket. Returns {"raw": [str], "new_tokens": [int], "seconds": [float]}.

    For each ticket text in `tickets.text`:
      1. prompt = tok.apply_chat_template(build_messages(text), tokenize=False,
                                          add_generation_prompt=True)
      2. ids = tok(prompt, return_tensors="pt").to(device)
      3. sync(); t0 = time.perf_counter()
         out = model.generate(**ids, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.eos_token_id, **GREEDY)   -- inside torch.no_grad()
         sync(); seconds = time.perf_counter() - t0
      4. new = out[0][ids["input_ids"].shape[1]:]
         raw = tok.decode(new, skip_special_tokens=True).strip()
      5. append raw, len(new) and seconds to the three lists
    """
    raw_list = []
    new_tokens_list = []
    seconds_list = []

    for text in tickets.text:
        prompt = tok.apply_chat_template(build_messages(text), tokenize=False,
                                          add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt").to(device)

        sync()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=max_new_tokens,
                                  pad_token_id=tok.eos_token_id, **GREEDY)
        sync()
        seconds = time.perf_counter() - t0

        new = out[0][ids["input_ids"].shape[1]:]
        raw = tok.decode(new, skip_special_tokens=True).strip()

        raw_list.append(raw)
        new_tokens_list.append(len(new))
        seconds_list.append(seconds)

    return {"raw": raw_list, "new_tokens": new_tokens_list, "seconds": seconds_list}


In [16]:
# --- provided: all three models over all sixty tickets ----------------------------------------
gen_results = {}
for _name in MODELS:
    _tok, _model = load_model(_name)
    _out = attempt("classify_all", classify_all, _model, _tok)
    free_model(_model); del _tok, _model
    if _out is None:
        gen_results = None
        break
    gen_results[_name] = _out
    print(f"{_name:42s} {sum(_out['seconds']):6.1f} s   first replies: {_out['raw'][:4]}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-360M-Instruct          41.9 s   first replies: ['Shipping', 'Shipping', 'Shipping', 'Yes, you can change the delivery address']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen/Qwen2.5-0.5B-Instruct                   36.2 s   first replies: ['Tracking', 'Delivery address', 'Shipping', 'Billing']


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen/Qwen2.5-1.5B-Instruct                  128.4 s   first replies: ['Shipping', 'Shipping', 'Shipping', 'Shipping']


In [17]:
# --- provided: the metrics table and the per-queue table ---------------------------------------

def summarise(out: dict) -> dict:
    preds = [parse_label(r) for r in out["raw"]]
    ok = [p == g for p, g in zip(preds, GOLD)]
    diff = tickets.difficulty.tolist()
    return {
        "accuracy": round(float(np.mean(ok)), 3),
        "invalid replies": round(float(np.mean([p is None for p in preds])), 3),
        "acc clear": round(float(np.mean([o for o, d in zip(ok, diff) if d == "clear"])), 3),
        "acc ambiguous": round(float(np.mean([o for o, d in zip(ok, diff) if d == "ambiguous"])), 3),
        "mean new tokens": round(float(np.mean(out["new_tokens"])), 2),
        "s per ticket": round(float(np.mean(out["seconds"])), 3),
    }


if not need(gen_results):
    metrics = pd.DataFrame([{"model": n.split("/")[-1], **summarise(o)} for n, o in gen_results.items()])
    print(metrics.to_string(index=False))

    # accuracy per queue: the table the lecture said not to skip -- one overall number
    # hides a rare-queue failure
    per_queue = {}
    for n, o in gen_results.items():
        preds = [parse_label(r) for r in o["raw"]]
        per_queue[n.split("/")[-1]] = {
            c: round(float(np.mean([p == g for p, g in zip(preds, GOLD) if g == c])), 2) for c in CATEGORIES}
    print("\naccuracy per queue (10 tickets each)\n")
    print(pd.DataFrame(per_queue).T.to_string())


                model  accuracy  invalid replies  acc clear  acc ambiguous  mean new tokens  s per ticket
SmolLM2-360M-Instruct     0.133            0.250      0.119          0.167             4.72         0.698
Qwen2.5-0.5B-Instruct     0.450            0.083      0.476          0.389             2.38         0.604
Qwen2.5-1.5B-Instruct     0.633            0.000      0.738          0.389             2.15         2.140

accuracy per queue (10 tickets each)

                       Shipping  Billing  Returns  Defect  Account  Other
SmolLM2-360M-Instruct       0.8      0.0      0.0     0.0      0.0    0.0
Qwen2.5-0.5B-Instruct       0.5      0.3      0.8     0.9      0.2    0.0
Qwen2.5-1.5B-Instruct       0.7      0.8      1.0     0.8      0.4    0.1


In [18]:
# --- provided: the confusion matrix and every wrong reply, for your best local model ---------
if not need(gen_results):
    BEST_LOCAL = MODELS[-1]          # change this if a different model scored highest
    best = [parse_label(r) for r in gen_results[BEST_LOCAL]["raw"]]
    filled = [p if p is not None else "<invalid>" for p in best]
    labels = CATEGORIES + ["<invalid>"]
    cm = confusion_matrix(GOLD, filled, labels=labels)
    print(f"Confusion matrix -- {BEST_LOCAL}  (rows = gold queue, columns = predicted)\n")
    print(pd.DataFrame(cm, index=labels, columns=labels).loc[CATEGORIES].to_string())

    print("\n\nEvery ticket it got wrong:\n")
    for i, (p, g) in enumerate(zip(filled, GOLD)):
        if p != g:
            t = tickets.iloc[i]
            print(f"{t.id}  {t.difficulty:<9s} gold={g:<9s} pred={p:<10s} raw={gen_results[BEST_LOCAL]['raw'][i]!r}")
            print(f"        {t.text[:100]}")


Confusion matrix -- Qwen/Qwen2.5-1.5B-Instruct  (rows = gold queue, columns = predicted)

          Shipping  Billing  Returns  Defect  Account  Other  <invalid>
Shipping         7        1        1       1        0      0          0
Billing          0        8        0       0        2      0          0
Returns          0        0       10       0        0      0          0
Defect           0        0        2       8        0      0          0
Account          0        3        2       0        4      1          0
Other            2        0        7       0        0      1          0


Every ticket it got wrong:

T007  clear     gold=Shipping  pred=Returns    raw='Returns'
        My order shipped in two boxes and only one arrived. The sleeping bag is missing.
T008  ambiguous gold=Shipping  pred=Billing    raw='Billing'
        I was charged for expedited shipping but the package took nine days. I'd like the shipping fee back.
T010  ambiguous gold=Shipping  pred=Defect     raw='Defe

### Your answer — Task 3

Accuracy rose from 0.133 (SmolLM2-360M) to 0.450 (Qwen2.5-0.5B), which is a gain of 0.317, or about 19 tickets. It then rose to 0.633 (Qwen2.5-1.5B), a gain of 0.183, or about 11 tickets. The larger jump in parameters bought less accuracy, though the first step also crosses model families and includes the 25% invalid replies from SmolLM2. The gain from 0.5B to 1.5B came from clear tickets (0.476 to 0.738), while ambiguous accuracy stayed at 0.389 for both.

The hardest queue for the 1.5B model is Other, at 0.1. Seven of the ten Other tickets went to Returns, which includes clear ones like T051 (sponsorship), T052 (store hours, sent to Shipping) and T055 (pre-sale tent question, raw reply 'Returns'). Under the desk rules, sponsorship, store hours and prepurchase advice are Other, so these are model failures, the model rarely picks Other and defaults to a substantive queue. The ambiguous misses look different, T038 ("the stove failed... I want my money back") and T040 (bent trekking pole, wants to send it back) were both answered 'Returns', but Defect beats Returns. The system prompt never states that tie break rule, so those errors reflect the prompt and category scheme as much as the model.

The smallest model is not the fastest per ticket. SmolLM2-360M took 0.941s per ticket, Qwen2.5-0.5B took 0.769s, and Qwen2.5-1.5B took 2.040s. Every new token is a full pass through the model, so speed depends on how many tokens are written as well as on model size. SmolLM2 writes about 4.72 new tokens per ticket on average vs about 2.38 for Qwen 0.5B, and its 25% invalid replies are rambling answers like "Yes, you can change the delivery address" that ran until the 8 token cap. Both Qwen models write about two tokens, so their times follow model size, the 1.5B model is about 2.65x slower than the 0.5B, close to its 3.1x larger parameter count.

The more expensive mistake is Defect to Returns (2 tickets in the matrix), the desk says the Defect queue owns the warranty decision, so routing it to Returns loses the fault report. A rare but more serious one is T049, an account takeover report with an order the customer never placed, sent to Returns instead of Account, where a security problem waits in the wrong queue. The most frequent error, Other to Returns (7 tickets), is cheaper per ticket, since someone can answer it quickly.


---

## Task 4 — Small vs large, and your recommendation

*Lecture slides: **Small vs large is an engineering decision** and the wrap-up case,
**Design Northwind's support workflow**. Notebook §6.4–6.5.*

You do not have an API key yet — those arrive in Week 6 — so the frontier side is supplied
as a **fixture**: a reply string for each of the 60 tickets, scored exactly the way the local
models are scored. The reply strings were **authored, not sampled from a live model**; the
token counts are real, so the cost arithmetic is real, but the accuracy it produces is an
illustration of the procedure, not a measurement of any product. Your memo must say so.
The fixture file, `frontier_cached.json`, loads from the course's public data repository
the same way the tickets do.


In [19]:
# --- provided: load the frontier fixture --------------------------------------
import urllib.request

_fp = data_path("frontier_cached.json")
if Path(_fp).exists():
    frontier = json.loads(Path(_fp).read_text())
else:
    with urllib.request.urlopen(_fp) as fh:
        frontier = json.loads(fh.read())

print(json.dumps(frontier["meta"], indent=2))
frontier_raw = [frontier["responses"][i]["reply"] for i in tickets.id]
print("\nfirst eight replies:", frontier_raw[:8])


{
  "kind": "fixture",
  "model_label": "frontier-mini (stand-in)",
  "note": "Authored reply strings, NOT sampled from a live model. Token counts are real tiktoken o200k_base counts over the exact strings above. No latency is recorded because none was measured. Set OPENAI_API_KEY and re-run the notebook's live cell to replace this with a real transcript.",
  "encoding": "o200k_base",
  "system_prompt_sha256": "9a64918d1a1a39199253a5ec1e438b16da40183e20e5d326ba26ed22af4b9839",
  "system_prompt_tokens": 154,
  "n_tickets": 60,
  "n_agreeing_with_gold": 52
}

first eight replies: ['Shipping', 'Shipping', 'Shipping.', 'Shipping', 'Shipping', 'Shipping', 'Shipping', 'Billing']


In [20]:
# --- 4a: the API bill ------------------------------------------------------------
# List prices for a frontier "mini" tier, in US dollars per MILLION tokens. Inputs to your
# analysis, not facts about the universe: check the provider's pricing page and record the
# date you checked in your memo.
PRICE_IN_PER_M = 0.15
PRICE_OUT_PER_M = 0.60
TICKETS_PER_WEEK = 2400


def usd_cost(input_tokens: int, output_tokens: int,
             price_in_per_m: float = PRICE_IN_PER_M, price_out_per_m: float = PRICE_OUT_PER_M) -> float:
    """Dollar cost of a request or a batch: tokens / 1,000,000 x price, for input and output."""
    return (input_tokens / 1e6) * price_in_per_m + (output_tokens / 1e6) * price_out_per_m


_in = sum(v["prompt_tokens"] for v in frontier["responses"].values())
_out = sum(v["completion_tokens"] for v in frontier["responses"].values())
_c60 = attempt("usd_cost", usd_cost, _in, _out)
if _c60 is not None:
    per_ticket_usd = _c60 / len(tickets)
    print(f"fixture totals: {_in:,} input tokens, {_out:,} output tokens for {len(tickets)} tickets")
    print(f"cost per ticket : ${per_ticket_usd:.6f}")
    print(f"cost per week   : ${per_ticket_usd * TICKETS_PER_WEEK:.2f}   ({TICKETS_PER_WEEK:,} tickets)")
    print(f"cost per year   : ${per_ticket_usd * TICKETS_PER_WEEK * 52:.2f}")


fixture totals: 10,351 input tokens, 76 output tokens for 60 tickets
cost per ticket : $0.000027
cost per week   : $0.06   (2,400 tickets)
cost per year   : $3.32


In [21]:
# --- provided: the comparison table -------------------------------------------------------
# The local machine is already in the ops room, so its line is machine time, not dollars:
# seconds per ticket x 2,400 tickets, in minutes per week.
if not need(gen_results, _c60):
    rows = []
    for n, o in gen_results.items():
        m = summarise(o)
        rows.append({"system": n.split("/")[-1], "where the text goes": "the ops-room machine",
                     "accuracy": m["accuracy"], "acc clear": m["acc clear"], "acc ambiguous": m["acc ambiguous"],
                     "invalid replies": m["invalid replies"], "s per ticket": m["s per ticket"],
                     "cost per week": f"{m['s per ticket'] * TICKETS_PER_WEEK / 60:.1f} machine-minutes"})
    f_out = {"raw": frontier_raw, "new_tokens": [frontier["responses"][i]["completion_tokens"] for i in tickets.id],
             "seconds": [float("nan")] * len(tickets)}
    fm = summarise(f_out)
    rows.append({"system": "frontier (FIXTURE)", "where the text goes": "a third party's servers",
                 "accuracy": fm["accuracy"], "acc clear": fm["acc clear"], "acc ambiguous": fm["acc ambiguous"],
                 "invalid replies": fm["invalid replies"], "s per ticket": "not measured",
                 "cost per week": f"${per_ticket_usd * TICKETS_PER_WEEK:.2f}"})
    comparison = pd.DataFrame(rows)
    print(comparison.to_string(index=False))
    print("\nThe frontier row's accuracy is a fixture and its latency was never measured. Say both in the memo.")


               system     where the text goes  accuracy  acc clear  acc ambiguous  invalid replies s per ticket        cost per week
SmolLM2-360M-Instruct    the ops-room machine     0.133      0.119          0.167            0.250        0.698 27.9 machine-minutes
Qwen2.5-0.5B-Instruct    the ops-room machine     0.450      0.476          0.389            0.083        0.604 24.2 machine-minutes
Qwen2.5-1.5B-Instruct    the ops-room machine     0.633      0.738          0.389            0.000         2.14 85.6 machine-minutes
   frontier (FIXTURE) a third party's servers     0.867      0.976          0.611            0.000 not measured                $0.06

The frontier row's accuracy is a fixture and its latency was never measured. Say both in the memo.


### Your memo — Task 4

Choose: I recommend buying the API, running a frontier mini model at greedy/temperature 0 with a person approving the risky queues, and not deploying a local model. Legal must sign off on sending ticket text to a third party, I also recommend stripping order numbers, card digits and addresses before it leaves the building. Cost is not the argument for going local. On the fixture, the API costs about $0.06 a week ($3.32 a year for 2,400 tickets a week), using the list prices in this notebook ($0.15 in, $0.60 out per million tokens). The best local model, Qwen2.5-1.5B-Instruct, fits on the ops room machine (6.17 GB) and needs about needs about 86 machine-minutes a week (2.140 s per ticket). But it only has 0.633 accuracy, which would misroute a bunch of tickets.

Evaluate: The measured numbers are the local ones, 0.633 overall for 1.5B, 0.738 on clear tickets, 0.389 on ambiguous ones, 0.1 on the Other queue, and 0 invalid replies (SmolLM2-360M had 0.250). The frontier numbers are fixture numbers, the replies were authored, not sampled from a live model, and no latency was measured. So its 0.867 accuracy (0.976 clear, 0.611 ambiguous) shows the procedure and is not evidence about any product. It's token counts are accurate, so the cost arithmetic holds. The errors that matter most are Defect sent to Returns, which loses the warranty fault report, and account security reports sent to the wrong queue. 60 tickets, ten per queue, can show direction but cannot support per queue claims, and it says nothing about the real mix of traffic. The next measurement is to run the live API on these same 60 tickets, with the tie break rules added to the system prompt.

Control: A person approves every Defect prediction, every Account ticket mentioning unauthorised access, and every prediction of Other, since even the best local model handled that queue badly. Any invalid reply goes to a human queue, and a weekly sample of auto routed tickets is audited. The recommendation flips to a local model, with a person deciding every ticket, if Legal refuses third party processing even after redaction. It flips if the live test shows the API is below the fixture's 0.867, especially if most real traffic is ambiguous. It also flips if a local model given the tie break rules approaches API accuracy.
